# Transaction fees and capitalisation vs non-capitalisation

See https://support.lusid.com/docs/capitalising-or-expensing-transaction-fees.

In summary: three portfolios, each with the same purchase transaction loaded:

* Units: 10
* Price: 10
* Gross consideration: 100
* Total fees: 10
* Total consideration: gross consideration plus total fees = 110  **<-- this is the cost of the cash movement**
* Trade amount: gross consideration plus total capitalised fees = &lt;depends on portfolio&gt;   **<-- this is the cost of the stock movement**

Each portfolio has different fee capitalisation choices, which should yield different trade amounts:

<table>
  <thead>
    <tr>
      <th></th>
      <th style="text-align:center">Broker fee (£5)</th>
      <th style="text-align:center">Exchange fee (£2)</th>
      <th style="text-align:center">Clearing fee (£2.50)</th>
      <th style="text-align:center">NFA fee (£0.50)</th>
      <th style="text-align:center">Total fees</th>
      <th style="text-align:center">Total cap fees</th>
      <th style="text-align:center">Total non-cap fees</th>
      <th style="text-align:center">Expected trade amount</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><strong>AllCap portfolio</strong></td><td style="text-align:center">C</td><td style="text-align:center">C</td><td style="text-align:center">C</td><td style="text-align:center">C</td><td style="text-align:center">£10</td><td style="text-align:center">£10</td><td style="text-align:center"></td><td style="text-align:center">£110</td></tr>
    <tr><td><strong>AllNonCap portfolio</strong></td><td style="text-align:center">NC</td><td style="text-align:center">NC</td><td style="text-align:center">NC</td><td style="text-align:center">NC</td><td style="text-align:center">£10</td><td style="text-align:center"></td><td style="text-align:center">£10</td><td style="text-align:center">£100</td></tr>
    <tr><td><strong>Mixed portfolio</strong></td><td style="text-align:center">C</td><td style="text-align:center">C</td><td style="text-align:center">NC</td><td style="text-align:center">NC</td><td style="text-align:center">£10</td><td style="text-align:center">£7</td><td style="text-align:center">£3</td><td style="text-align:center">£107</td></tr>
  </tbody>
</table>

## Setup

In [1]:
import os
import pandas as pd
import numpy as np
import json
import uuid
from IPython.core.display import HTML
import logging
from datetime import datetime, timezone, timedelta
logging.basicConfig(level = logging.INFO)

import finbourne.sdk.services.lusid.api as la
import finbourne.sdk.services.lusid.models as lm

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.lpt.lpt import to_date

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', 200)
pd.options.display.float_format = "{:,.2f}".format

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)
    
# Confirm success by printing SDK version
api_status = pd.DataFrame(api_factory.build(la.ApplicationMetadataApi).get_lusid_versions().to_dict())
display(api_status)

,apiVersion,buildVersion,excelVersion,links
0,v0,0.6.16204.0,0.5.3666,"{'relation': 'RequestLogs', 'href': 'https://jamleed.lusid.com/app/insights/logs/2026072107-5f672dd09c824164b78610ab8c1340fa', 'description': 'A link to the LUSID Insights website showing all logs..."


In [2]:
# Build all the required APIs
try:
    instruments_api = api_factory.build(la.InstrumentsApi)
    aggregation_api = api_factory.build(la.AggregationApi)
    recipe_api = api_factory.build(la.ConfigurationRecipeApi)
    quotes_api = api_factory.build(la.QuotesApi)
    property_definition_api = api_factory.build(la.PropertyDefinitionsApi)
    transaction_portfolios_api = api_factory.build(la.TransactionPortfoliosApi)
    portfolios_api = api_factory.build(la.PortfoliosApi)
    abor_api = api_factory.build(la.AborApi)
    abor_config_api = api_factory.build(la.AborConfigurationApi)
    chart_accts_api = api_factory.build(la.ChartOfAccountsApi)
    transaction_config_api = api_factory.build(la.TransactionConfigurationApi)
    fund_api = api_factory.build(la.FundsApi)
    feetypes_api = api_factory.build(la.FeeTypesApi)
    fundconfig_api = api_factory.build(la.FundConfigurationApi)
    transaction_fees_api = api_factory.build(la.TransactionFeeTypesApi)
    print("All APIs built correctly")
except ApiException as e:
    print(e)

All APIs built correctly


## Create a scope and code for entities in the Notebook

Keep data segregated from other data in LUSID.

In [3]:
module_scope = "FBNTutorials"
module_code = "TransactionFeeEngine1"
print(f"'{module_scope}\\{module_code}' scope and code created.")

'FBNTutorials\TransactionFeeEngine1' scope and code created.


## Create property types

In [4]:
# Create convenience function for creating property types
def create_property_type(property_domain, property_scope, property_code, data_type):
    
    # Define property type with a scope and code unique to the domain
    property_type_request = lm.CreatePropertyDefinitionRequest(
        domain = property_domain,
        scope = property_scope,
        code = property_code,
        display_name = property_code,
        data_type_id = lm.ResourceId(scope = "system", code = data_type)
    )
    
    # Create property type in LUSID
    try:
        property_type_response = property_definition_api.create_property_definition(
            create_property_definition_request = property_type_request
        )
        print(f"Property type created with the following key: {property_type_response.key}")
        return property_type_response.key
    except ApiException as e:
        if json.loads(e.body)["name"] == "PropertyAlreadyExists":
            logging.info(
                f"Property type with the following key already exists: {property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"
            )  
        return f"{property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"

### Create `Transaction` output properties to store transaction fee calculation results

In [5]:
broker_fee_output_property = create_property_type("Transaction", "Fees", "Broker", "currencyAndAmount")
print(broker_fee_output_property)

INFO:root:Property type with the following key already exists: Transaction/Fees/Broker


Transaction/Fees/Broker


In [6]:
exchange_fee_output_property = create_property_type("Transaction", "Fees", "Exchange", "currencyAndAmount")
print(exchange_fee_output_property)

INFO:root:Property type with the following key already exists: Transaction/Fees/Exchange


Transaction/Fees/Exchange


In [7]:
clearing_fee_output_property = create_property_type("Transaction", "Fees", "Clearing", "currencyAndAmount")
print(clearing_fee_output_property)

INFO:root:Property type with the following key already exists: Transaction/Fees/Clearing


Transaction/Fees/Clearing


In [8]:
nfa_fee_output_property = create_property_type("Transaction", "Fees", "NFA", "currencyAndAmount")
print(nfa_fee_output_property)

INFO:root:Property type with the following key already exists: Transaction/Fees/NFA


Transaction/Fees/NFA


### Create `Portfolio` property to determine capitalisation fee status for all transactions in a portfolio

In [9]:
portfolio_fee_status_property = create_property_type("Portfolio", "Fees", "Status", "string")
print(portfolio_fee_status_property)

INFO:root:Property type with the following key already exists: Portfolio/Fees/Status


Portfolio/Fees/Status


## Create transaction fee types

See https://support.lusid.com/docs/how-do-i-create-a-transaction-fee-type

In [10]:
def list_transaction_fee_types():
    list_fees_response = transaction_fees_api.list_transaction_fee_types(
        filter = f"id.scope eq '{module_scope}' and id.code startswith '{module_code}'"
    )
    list_fees_response_df = lusid_response_to_data_frame(list_fees_response)
    list_fees_response_df.drop(list_fees_response_df.filter(regex='version').columns, axis=1, inplace=True)
    display(list_fees_response_df)

list_transaction_fee_types()

""


In [11]:
def create_transaction_fee_type(fee_type, fee_percent):

    create_fee_request = lm.CreateTransactionFeeTypeRequest(
        displayName = f"{fee_type} fee type",
        description=f"A {fee_type} fee type",
        calculation=lm.FeeCalculationRequest(
            formula=f"Properties[Transaction/default/GrossConsideration] * ({fee_percent} / 100)"
        ),
        condition="Transaction.units gte 10",
        txnPropertyKey=f"Transaction/Fees/{fee_type}"
    )
    
    try:
        create_fee_response = transaction_fees_api.create_transaction_fee_type(
            scope = module_scope,
            code = f"{module_code}-{fee_type}",
            create_transaction_fee_type_request=create_fee_request
        )
        print("Success")
    except ApiException as e:
        print(e.body)

In [12]:
create_transaction_fee_type("Broker", 5)
create_transaction_fee_type("Exchange", 2)
create_transaction_fee_type("Clearing", 2.5)
create_transaction_fee_type("NFA", 0.5)

list_transaction_fee_types()

Success
Success
Success
Success


,id.scope,id.code,display_name,description,calculation.formula,condition,txn_property_key,properties,is_active
0,FBNTutorials,TransactionFeeEngine1-Broker,Broker fee type,A Broker fee type,Properties[Transaction/default/GrossConsideration] * (5 / 100),Transaction.units gte 10,Transaction/Fees/Broker,{},True
1,FBNTutorials,TransactionFeeEngine1-Exchange,Exchange fee type,A Exchange fee type,Properties[Transaction/default/GrossConsideration] * (2 / 100),Transaction.units gte 10,Transaction/Fees/Exchange,{},True
2,FBNTutorials,TransactionFeeEngine1-Clearing,Clearing fee type,A Clearing fee type,Properties[Transaction/default/GrossConsideration] * (2.5 / 100),Transaction.units gte 10,Transaction/Fees/Clearing,{},True
3,FBNTutorials,TransactionFeeEngine1-NFA,NFA fee type,A NFA fee type,Properties[Transaction/default/GrossConsideration] * (0.5 / 100),Transaction.units gte 10,Transaction/Fees/NFA,{},True


## Create a transaction type that handles fees

In [13]:
def check_TT(tt, scope):
    try:
        tt_response = transaction_config_api.get_transaction_type(source = f"default", type = tt, scope=scope)
        print(f"\n{tt} transaction type:")
        display(lusid_response_to_data_frame(tt_response.aliases))
        display(lusid_response_to_data_frame(tt_response.movements))
        display(lusid_response_to_data_frame(tt_response.calculations))
    except ApiException as e:
        print(e.body)
        
def check_side(side, scope):
    try:
        side_response = transaction_config_api.get_side_definition(scope = scope, side = side)
        print(f"\n{side} side:")
        side_response_df = lusid_response_to_data_frame(side_response).transpose()
        side_response_df.drop(side_response_df.filter(regex='links').columns, axis=1, inplace=True)
        display(side_response_df)  
    except ApiException as e:
        print(e.body)

### Create sides

Must be created before transaction types. Includes recreating the built-in `Side1` and `Side2` in the custom transaction type scope.

In [14]:
# Recreate Side1 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TradeAmount"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [15]:
# Recreate Side2 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:SettleCcy",
    currency = "Txn:SettlementCurrency",
    rate = "SettledToPortfolioRate",
    units = "Txn:TotalConsideration",
    amount = "Txn:TotalConsideration"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side2",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [16]:
# Define a custom `CarryNonCapFees` side to handle carry for non-capitalised fees
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:TotalNonCapitalisedFees",
    amount = "Txn:TotalNonCapitalisedFees",
)

try:
    response = transaction_config_api.set_side_definition(
        side = "CarryNonCapFees",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e.body)

Success


### Create `BuyWithFees` transaction type for equity purchases

See https://support.lusid.com/docs/what-is-a-transaction-type-calculation#fee-calculation for more on the `Fee` calculation type.

All created in a custom transaction type scope, which must be registered with the portfolio in which transactions are loaded.

In [17]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "BuyWithFees",
            description = "Type for automatic calculation of capitalised/expensed fees",
            transaction_class = "Trading",
            transaction_roles = "Longer",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            direction = 1,
            side = "Side1",
        ),
        lm.TransactionTypeMovement(
            movement_types = "CashCommitment",
            direction = -1,
            side = "Side2",
        ),
        lm.TransactionTypeMovement(
            movement_types = "Carry",
            direction = -1,
            side = "CarryNonCapFees",
        )
    ],
    calculations = [
        lm.TransactionTypeCalculation(
            type = "Txn:GrossConsideration",
        ),
        # The cost of the cash movement must be equal to the total outlay (ie. all the fees, cap and non-cap)
        lm.TransactionTypeCalculation(
            type = "DeriveTotalConsideration",
            formula="Txn:GrossConsideration + Txn:TotalFees"

        ),
        lm.TransactionTypeCalculation(
            type = "Fee",
            transactionFeeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-Broker"
            ),
            # Capitalised if portfolio fee status is "Mixed" or "AllCap", otherwise non-capitalised
            transactionFeeCapitalisation=lm.TransactionFeeCapitalisation(
                capitalisation="Conditional",
                capitalisedCondition="Portfolio.Properties[Portfolio/Fees/Status] neq 'AllNonCap'"
            )
        ),
        lm.TransactionTypeCalculation(
            type = "Fee",
            transactionFeeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-Exchange"
            ),
            # Capitalised if portfolio fee status is "Mixed" or "AllCap", otherwise non-capitalised
            transactionFeeCapitalisation=lm.TransactionFeeCapitalisation(
                capitalisation="Conditional",
                capitalisedCondition="Portfolio.Properties[Portfolio/Fees/Status] neq 'AllNonCap'"
            )
        ),
        lm.TransactionTypeCalculation(
            type = "Fee",
            transactionFeeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-Clearing"
            ),
            # Capitalised if portfolio fee status is "AllCap", otherwise non-capitalised
            transactionFeeCapitalisation=lm.TransactionFeeCapitalisation(
                capitalisation="Conditional",
                capitalisedCondition="Portfolio.Properties[Portfolio/Fees/Status] eq 'AllCap'"
            )
        ),
        lm.TransactionTypeCalculation(
            type = "Fee",
            transactionFeeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-NFA"
            ),
            # Capitalised if portfolio fee status is "AllCap", otherwise non-capitalised
            transactionFeeCapitalisation=lm.TransactionFeeCapitalisation(
                capitalisation="Conditional",
                capitalisedCondition="Portfolio.Properties[Portfolio/Fees/Status] eq 'AllCap'"
            )
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "BuyWithFees",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e.body)

Success


In [18]:
check_TT("BuyWithFees", f"{module_scope}{module_code}")
check_side("Side1", f"{module_scope}{module_code}")
check_side("Side2", f"{module_scope}{module_code}")
check_side("CarryNonCapFees", f"{module_scope}{module_code}")


BuyWithFees transaction type:


,type,description,transaction_class,transaction_roles,is_default
0,BuyWithFees,Type for automatic calculation of capitalised/expensed fees,Trading,Longer,False


,movement_types,side,direction,properties,mappings,movement_options,condition,settlement_mode
0,StockMovement,Side1,1,{},[],[],,Internal
1,CashCommitment,Side2,-1,{},[],[],,Internal
2,Carry,CarryNonCapFees,-1,{},[],[],,Internal


,type,formula,transaction_fee_id.scope,transaction_fee_id.code,transaction_fee_capitalisation.capitalisation,transaction_fee_capitalisation.capitalised_condition
0,Txn:GrossConsideration,None,NaN,NaN,NaN,NaN
1,DeriveTotalConsideration,Txn:GrossConsideration + Txn:TotalFees,NaN,NaN,NaN,NaN
2,Fee,None,FBNTutorials,TransactionFeeEngine1-Broker,Conditional,Portfolio.Properties[Portfolio/Fees/Status] neq 'AllNonCap'
3,Fee,None,FBNTutorials,TransactionFeeEngine1-Exchange,Conditional,Portfolio.Properties[Portfolio/Fees/Status] neq 'AllNonCap'
4,Fee,None,FBNTutorials,TransactionFeeEngine1-Clearing,Conditional,Portfolio.Properties[Portfolio/Fees/Status] eq 'AllCap'
5,Fee,None,FBNTutorials,TransactionFeeEngine1-NFA,Conditional,Portfolio.Properties[Portfolio/Fees/Status] eq 'AllCap'



Side1 side:


,side,security,currency,rate,units,amount,notional_amount,current_face,scope
response_values,Side1,Txn:LusidInstrumentId,Txn:TradeCurrency,Txn:TradeToPortfolioRate,Txn:Units,Txn:TradeAmount,0,None,None



Side2 side:


,side,security,currency,rate,units,amount,notional_amount,current_face,scope
response_values,Side2,Txn:SettleCcy,Txn:SettlementCurrency,SettledToPortfolioRate,Txn:TotalConsideration,Txn:TotalConsideration,0,None,None



CarryNonCapFees side:


,side,security,currency,rate,units,amount,notional_amount,current_face,scope
response_values,CarryNonCapFees,Txn:LusidInstrumentId,Txn:TradeCurrency,Txn:TradeToPortfolioRate,Txn:TotalNonCapitalisedFees,Txn:TotalNonCapitalisedFees,0,None,None


## Master an equity instrument

In [19]:
def master_instrument(assetclass, id, currency):
    
    if assetclass == "equity":
        instrument_request = {
            id: lm.InstrumentDefinition(
                name = id,
                identifiers = {
                    "ClientInternal": lm.InstrumentIdValue(value=id),
                },
                definition = lm.Equity(instrument_type = "Equity", dom_ccy = currency),
            )
        }
    
    try:
        instrument_response = instruments_api.upsert_instruments(
            request_body = instrument_request,
            scope = f"{module_scope}{module_code}"
        )
        # Return LUID from (only) instrument object
        return list(instrument_response.values.values())[0].lusid_instrument_id 
    except ApiException as e:
        print(e.body)

In [20]:
luid_dict = {}
luid_dict["Tesco"] = master_instrument("equity", "Tesco", "GBP")
for k, v in luid_dict.items():
    print(f"{k}: {v}")

Tesco: LUID_00003H3N


In [21]:
def list_instrs():
    instr_response = instruments_api.list_instruments(scope=f"{module_scope}{module_code}")
    instr_response_df = lusid_response_to_data_frame(instr_response, use_camel_case=True)
    instr_response_df.drop(instr_response_df.filter(regex='version|href|staged').columns, axis=1, inplace=True)
    display(instr_response_df.transpose())

list_instrs()

,0
scope,FBNTutorialsTransactionFeeEngine1
lusidInstrumentId,LUID_00003H3N
name,Tesco
identifiers.LusidInstrumentId,LUID_00003H3N
identifiers.ClientInternal,Tesco
properties,[]
instrumentDefinition.instrumentType,Equity
instrumentDefinition.identifiers,{}
instrumentDefinition.domCcy,GBP
instrumentDefinition.lotSize,1


## Set up GBP transaction portfolios

* AllCap portfolio: All fees capitalised
* AllNonCap portfolio: All fees non-capitalised
* Mixed portfolio: Exchange and Broker fees capitalised, Clearing and NFA non-capitalised.

In [22]:
ports = ["AllCap", "AllNonCap", "Mixed"]
# ports = ["Mixed"]

In [23]:
### Create portfolio recipe (also used as valuation recipe)
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as the portfolio
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-SimpleStatic",
    description = "A recipe to value an equity",
    market = lm.MarketContext(
        market_rules = [
            # Look up FX spot rates in the LUSID quote store, if needed
            lm.MarketDataKeyRule(
                key = "Fx.CurrencyPair.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Rate",
                field = "mid",
                quote_interval = "1D.0D",
            ),
            # Rule for market prices for valuation (keyed by LusidInstrumentId)
            lm.MarketDataKeyRule(
                key = "Quote.LusidInstrumentId.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Price",
                field = "mid",
                quote_interval = "1D.0D",
            )
        ]
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e.body)

Success


In [24]:
def create_portfolio(port):
    portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name = f"{port} fees portfolio",
        code = f"{module_code}-{port}",
        # Set the portfolio currency
        base_currency = "GBP",
        # Must be before first transaction recorded
        created = datetime.strptime("2024-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        # Attempt to resolve transactions to instruments in the custom scope before falling back to the default scope
        instrument_scopes = [f"{module_scope}{module_code}"],
        # Register transaction type scope
        transactionTypeScope=f"{module_scope}{module_code}",
        # Register portfolio recipe        
        instrumentEventConfiguration=lm.InstrumentEventConfiguration(
            recipeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-SimpleStatic"
            )
        ),
        properties = {
            f"{portfolio_fee_status_property}": lm.ModelProperty(
                key = f"{portfolio_fee_status_property}",
                value = lm.PropertyValue(
                    label_value = port
                )
            )         
        }
    )

    try:
        portfolio_response=transaction_portfolios_api.create_portfolio(
            scope = module_scope,
            create_transaction_portfolio_request = portfolio_request
        )
        print(f"Portfolio with display name '{portfolio_response.display_name}' created effective {str(portfolio_response.created)}")
    except ApiException as e:
        print(e.body)

In [25]:
for port in ports:
    create_portfolio(port)

Portfolio with display name 'AllCap fees portfolio' created effective 2024-01-01 00:00:00+00:00
Portfolio with display name 'AllNonCap fees portfolio' created effective 2024-01-01 00:00:00+00:00
Portfolio with display name 'Mixed fees portfolio' created effective 2024-01-01 00:00:00+00:00


In [26]:
def get_port_details(port):
    portfolio_response = transaction_portfolios_api.get_details(
        scope = module_scope, 
        code = f"{module_code}-{port}",
    )
    portfolio_response_df = lusid_response_to_data_frame(portfolio_response).transpose()
    # Drop some noisy columns
    portfolio_response_df.drop(portfolio_response_df.filter(regex='version|href|staged|links|settlement').columns, axis=1, inplace=True)
    display(portfolio_response_df.transpose())

def get_port_properties(port):
    tp_property_response = portfolios_api.get_portfolio_properties(scope = module_scope, code = f"{module_code}-{port}")
    tp_property_response_df = lusid_response_to_data_frame(list(tp_property_response.properties.values()))
    display(tp_property_response_df)
    
for port in ports:
    print(f"\n{port} portfolio:")
    get_port_details(port)
    get_port_properties(port)


AllCap portfolio:


,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,TransactionFeeEngine1-AllCap
base_currency,GBP
corporate_action_source_id,None
sub_holding_keys,[]
instrument_scopes.0,FBNTutorialsTransactionFeeEngine1
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsTransactionFeeEngine1
cash_gain_loss_calculation_date,Default


,key,value.label_value,effective_from,effective_until
0,Portfolio/Fees/Status,AllCap,0001-01-01 00:00:00+00:00,9999-12-31 23:59:59.999999+00:00



AllNonCap portfolio:


,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,TransactionFeeEngine1-AllNonCap
base_currency,GBP
corporate_action_source_id,None
sub_holding_keys,[]
instrument_scopes.0,FBNTutorialsTransactionFeeEngine1
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsTransactionFeeEngine1
cash_gain_loss_calculation_date,Default


,key,value.label_value,effective_from,effective_until
0,Portfolio/Fees/Status,AllNonCap,0001-01-01 00:00:00+00:00,9999-12-31 23:59:59.999999+00:00



Mixed portfolio:


,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,TransactionFeeEngine1-Mixed
base_currency,GBP
corporate_action_source_id,None
sub_holding_keys,[]
instrument_scopes.0,FBNTutorialsTransactionFeeEngine1
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsTransactionFeeEngine1
cash_gain_loss_calculation_date,Default


,key,value.label_value,effective_from,effective_until
0,Portfolio/Fees/Status,Mixed,0001-01-01 00:00:00+00:00,9999-12-31 23:59:59.999999+00:00


## Load the same transaction in each portfolio

In [27]:
def create_transactions(port, txnid, tttype, luid, tradedate, settledate, quantity, price, ccy):
    
    create_txn_request = {
        "number_one": lm.TransactionRequest(
            transaction_id=txnid,
            type=tttype,
            instrument_identifiers = {"Instrument/default/LusidInstrumentId": luid},
            transaction_date=datetime.strptime(tradedate, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            settlement_date=datetime.strptime(settledate, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            units=quantity,
            # This is the market price, used for the gross consideration calculation in the transaction type
            transaction_price=lm.TransactionPrice(
                price=price, type="Price"
            ),
            # Total consideration set to 0 to trigger the calculation in the transaction type
            total_consideration = lm.CurrencyAndAmount(
                currency = ccy,
                amount = 0
            )
        )
    }
    
    try:
        create_txn_response = transaction_portfolios_api.batch_upsert_transactions(
            scope = f"{module_scope}",
            code = f"{module_code}-{port}",
            success_mode="Partial",
            request_body = create_txn_request
        )
        print(create_txn_response.failed) if create_txn_response.failed else print("Success")
    except ApiException as e:
        print(e.body)

In [28]:
for port in ports:
    create_transactions(port, "Txn01", "BuyWithFees", luid_dict["Tesco"], "2026-01-01 00:00:00", "2026-01-05 00:00:00", 10, 10, "GBP")

Success
Success
Success


# Examine impact on holdings

In [29]:
def get_portfolio_holdings(port, date):      
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))
    
    try:
        get_holdings_response = transaction_portfolios_api.get_holdings(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            property_keys=["Instrument/default/Name"]
        )
        get_holdings_response_df = lusid_response_to_data_frame(get_holdings_response)
        get_holdings_response_df.rename(columns = {
            "properties.Instrument/default/Name.value.label_value": "instrument"}, inplace = True)        
        # Drop some noisy columns
        get_holdings_response_df.drop(get_holdings_response_df.filter(regex='properties|sub_holding_keys|instrument_scope').columns, axis=1, inplace=True)
        display(get_holdings_response_df)
        # display(get_holdings_response_df.style.hide(axis="index").format("{:,.2f}", na_rep=""))
    except ApiException as e:
        print(e.body)

In [30]:
def get_output_transactions(port, start, end, show_summary):
    try:
        output_transactions_response = transaction_portfolios_api.build_transactions(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            transaction_query_parameters = lm.TransactionQueryParameters(
                start_date = datetime.strptime(start, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                end_date = datetime.strptime(end, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                properties = ["Instrument/default/name"]
            )
        )
        output_transactions_response_df = lusid_response_to_data_frame(output_transactions_response, use_camel_case=True)
        if show_summary == True:
            output_transactions_response_df = output_transactions_response_df[[
                "properties.Transaction/default/TotalFees.value.metricValue.value",
                "properties.Transaction/default/TotalCapitalisedFees.value.metricValue.value",
                "properties.Transaction/default/TotalNonCapitalisedFees.value.metricValue.value"
            ]]
            output_transactions_response_df.rename(columns = {
                "properties.Transaction/default/TotalFees.value.metricValue.value": "Transaction/default/TotalFees",
                "properties.Transaction/default/TotalCapitalisedFees.value.metricValue.value": "Transaction/default/TotalCapitalisedFees",
                "properties.Transaction/default/TotalNonCapitalisedFees.value.metricValue.value": "Transaction/default/TotalNonCapitalisedFees"
            }, inplace = True)        
            display(output_transactions_response_df)
        else:
            display(output_transactions_response_df.transpose())
    except ApiException as e:
        print(e.body)

In [31]:
for port in ports:
    print(f"\n\n{port} portfolio:\n")
    get_portfolio_holdings(port, "today")



AllCap portfolio:



,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,LUID_00003H3N,Tesco,P,10.00,10.00,110.00,GBP,110.00,GBP,GBP,Position,81630560,0.00,GBP,110.00,GBP,110.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,CCY_GBP,GBP,B,-110.00,-110.00,-110.00,GBP,-110.00,GBP,GBP,Balance,81630561,0.00,GBP,-110.00,GBP,-110.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00




AllNonCap portfolio:



,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,LUID_00003H3N,Tesco,P,10.00,10.00,100.00,GBP,100.00,GBP,GBP,Position,81630562,0.00,GBP,100.00,GBP,100.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,CCY_GBP,GBP,B,-110.00,-110.00,-110.00,GBP,-110.00,GBP,GBP,Balance,81630563,0.00,GBP,-110.00,GBP,-110.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00




Mixed portfolio:



,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,LUID_00003H3N,Tesco,P,10.00,10.00,107.00,GBP,107.00,GBP,GBP,Position,81630564,0.00,GBP,107.00,GBP,107.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,CCY_GBP,GBP,B,-110.00,-110.00,-110.00,GBP,-110.00,GBP,GBP,Balance,81630565,0.00,GBP,-110.00,GBP,-110.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00


In [32]:
for port in ports:
    print(f"\n{port} portfolio:\n")
    get_output_transactions(port, "2024-01-01", "2030-01-01", True)
    # get_output_transactions(port, "2024-01-01", "2030-01-01", False)


AllCap portfolio:



,Transaction/default/TotalFees,Transaction/default/TotalCapitalisedFees,Transaction/default/TotalNonCapitalisedFees
0,10.00,10.00,0.00



AllNonCap portfolio:



,Transaction/default/TotalFees,Transaction/default/TotalCapitalisedFees,Transaction/default/TotalNonCapitalisedFees
0,10.00,0.00,10.00



Mixed portfolio:



,Transaction/default/TotalFees,Transaction/default/TotalCapitalisedFees,Transaction/default/TotalNonCapitalisedFees
0,10.00,7.00,3.00


# Examine impact on valuation

In [33]:
def load_quotes(id_type, id, price, date, ccy, scale):
    if date == "today":
        #date = str(datetime.now().replace(microsecond=0))
        date = datetime.today().strftime('%Y-%m-%d 00:00:00')
        #print(date)

    quotes = {
        # Each quote must be upserted with an ephemeral key (uuid in this case), to track errors in the response
        str(uuid.uuid4()): lm.UpsertQuoteRequest(
            quote_id = lm.QuoteId(
                quote_series_id = lm.QuoteSeriesId(
                    # Must be one of the valid financial data vendor 'provider' values
                    provider = "Lusid",
                    instrument_id_type = id_type,
                    instrument_id = id,
                    quote_type = "Price",
                    # Case sensitive: the field value must match that of the equivalent recipe field exactly
                    field = "mid",
                ),
                effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            ),
            metric_value = lm.MetricValue(value = price, unit = ccy),
            scale_factor = scale,
        )
    }

    try:
        upsert_quotes_response = quotes_api.upsert_quotes(scope = f"{module_scope}{module_code}", request_body = quotes)    
        if upsert_quotes_response.failed == {}:
            print(f"{id} price for {date} successfully loaded into LUSID.")
        else:
            print(f"Some failures occurred. {len(upsert_quotes_response.failed)} prices did not get loaded into LUSID.")
    except ApiException as e:
        print(e.body)

In [34]:
load_quotes("LusidInstrumentId", luid_dict["Tesco"], 10, "2026-01-01 00:00:00", "GBP", 1)  # Transaction date, required for A2B
load_quotes("LusidInstrumentId", luid_dict["Tesco"], 10, "today", "GBP", 1)

LUID_00003H3N price for 2026-01-01 00:00:00 successfully loaded into LUSID.
LUID_00003H3N price for 2026-07-21 00:00:00 successfully loaded into LUSID.


In [35]:
def list_quotes():
    try:
        quotes_response = quotes_api.list_quotes_for_scope(f"{module_scope}{module_code}")
        quotes_response_df = lusid_response_to_data_frame(quotes_response)
        display(quotes_response_df)        
    except ApiException as e:
        print(e.body)
        
list_quotes()

,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,LUID_00003H3N,LusidInstrumentId,Price,mid,3ec79889-cd37-4038-a737-e085b4737288,2026-07-21T00:00:00.0000000+00:00,10.00,GBP,,,00u91lo2d7X42sdse2p7,2026-07-21 07:22:30.118154+00:00,1.00
1,Lusid,LUID_00003H3N,LusidInstrumentId,Price,mid,3ec79889-cd37-4038-a737-e085b4737288,2026-01-01T00:00:00.0000000+00:00,10.00,GBP,,,00u91lo2d7X42sdse2p7,2026-07-21 07:22:29.947212+00:00,1.00


In [36]:
def value_instruments(port, date, pnl_window):
    if date == "today":
        # #date = str(datetime.now().replace(microsecond=0))
        # date = datetime.today()
        date = datetime.today().strftime('%Y-%m-%d')

    valuation_request = lm.ValuationRequest(
        # Choose recipe to use
        recipe_id = lm.ResourceId(scope = module_scope, code = f"{module_code}-SimpleStatic"),
        # Specify metrics (also known as queryable keys) to report useful information
        metrics = [
            lm.AggregateSpec(key="Instrument/InstrumentCategory", op="Value"),
            lm.AggregateSpec(key="Instrument/default/Name", op="Value"),
            lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
            # lm.AggregateSpec(key="Valuation/Model/Name", op="Value"),
            # lm.AggregateSpec(key="Valuation/EffectiveAt", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Sum"),
            lm.AggregateSpec(key="Quotes/PriceOrFXRate", op="Value"),
            lm.AggregateSpec(key="Holding/Cost/Dom", op="Sum"),
            # lm.AggregateSpec(key="Valuation/CleanPV", op="Value"),
            lm.AggregateSpec(key="Valuation/PV", op="Sum"),
            # lm.AggregateSpec(key="Valuation/Accrued", op="Value"),
            # lm.AggregateSpec(key="Valuation/Exposure", op="Value"),          
            # lm.AggregateSpec(key="Valuation/CurrentNotional", op="Value"),
            lm.AggregateSpec(key="ProfitAndLoss/Total", op="Sum", options={"Window": f"{pnl_window}"}),      
            lm.AggregateSpec(key="ProfitAndLoss/Total/Market", op="Sum", options={"Window": f"{pnl_window}"}),
            # lm.AggregateSpec(key="ProfitAndLoss/Realised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            # lm.AggregateSpec(key="ProfitAndLoss/Unrealised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Total/Other", op="Sum", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="Aggregation/Errors", op="Value"), 
        ],
        # Identify portfolio to value
        portfolio_entity_ids = [lm.PortfolioEntityId(scope = module_scope, code = f"{module_code}-{port}")],
        valuation_schedule = lm.ValuationSchedule(effective_at = date),
        groupBy= ["Instrument/default/LusidInstrumentId"],
        #filters=[lm.PropertyFilter(left="Holding/default/Units", operator="NotEquals", right=0)]
    )

    try:
        # Get portfolio valuation
        val_response = aggregation_api.get_valuation(valuation_request = valuation_request)
        val_response_df = pd.json_normalize(val_response.to_dict()["data"], sep='.')
        #val_response_df = val_response_df.groupby("Instrument/default/LusidInstrumentId", as_index=False).sum(numeric_only=True)
        # Rename columns
        val_response_df.rename(
            columns = {
                "Instrument/InstrumentCategory": "Category",
                "Valuation/Model/Name": "Pricing model",
                "Instrument/default/LusidInstrumentId": "LUID",
                "Instrument/default/Name": "Name",
                "Valuation/EffectiveAt": "Date",
                "Holding/default/Units": "Units",
                "Sum(Holding/default/Units)": "Units",
                "Quotes/PriceOrFXRate": "Price",
                "Quotes/ScaleFactor": "Quote Scale Factor",
                "Holding/Cost/Dom": "Cost",
                "Sum(Holding/Cost/Dom)": "Cost",
                "Valuation/CleanPV": "Clean PV",
                "Valuation/PV": "PV",
                "Sum(Valuation/PV)": "PV",                
                "Valuation/Accrued": "Accrued Interest",
                "Valuation/Exposure": "Exposure",
                "Valuation/CurrentNotional": "Notional",            
                f"ProfitAndLoss/Total(Window=\"{pnl_window}\")": "Total P&L",
                f"Sum(ProfitAndLoss/Total(Window=\"{pnl_window}\"))": "Total P&L",
                f"ProfitAndLoss/Total/Market(Window=\"{pnl_window}\")": "Market P&L",
                f"Sum(ProfitAndLoss/Total/Market(Window=\"{pnl_window}\"))": "Market P&L",
                f"ProfitAndLoss/Realised/Market(Window=\"{pnl_window}\")": "Realised/Market P&L",
                f"ProfitAndLoss/Unrealised/Market(Window=\"{pnl_window}\")": "Unrealised/Market P&L",
                f"ProfitAndLoss/Total/Other(Window=\"{pnl_window}\")": "Other P&L",
                f"Sum(ProfitAndLoss/Total/Other(Window=\"{pnl_window}\"))": "Other P&L",
                "Aggregation/Errors": "Errors"
            },
            inplace = True,
        )
       # val_response_df["Date"] = pd.to_datetime(val_response_df["Date"]).dt.date
        display(val_response_df)
    except ApiException as e:
        print(e.body)

In [37]:
for port in ports:
    print(f"\n{port} portfolio:\n")
    value_instruments(port, "today", "YTD") # Random valuation date


AllCap portfolio:



,Category,Name,LUID,Units,Price,Cost,PV,Total P&L,Market P&L,Other P&L,Errors
0,Equity,Tesco,LUID_00003H3N,10.00,10.00,110.00,100.00,-10.00,-10.00,0.00,[]
1,Cash,GBP,CCY_GBP,-110.00,1.00,-110.00,-110.00,0.00,0.00,0.00,[]



AllNonCap portfolio:



,Category,Name,LUID,Units,Price,Cost,PV,Total P&L,Market P&L,Other P&L,Errors
0,Equity,Tesco,LUID_00003H3N,10.00,10.00,100.00,100.00,-10.00,0.00,-10.00,[]
1,Cash,GBP,CCY_GBP,-110.00,1.00,-110.00,-110.00,0.00,0.00,0.00,[]



Mixed portfolio:



,Category,Name,LUID,Units,Price,Cost,PV,Total P&L,Market P&L,Other P&L,Errors
0,Equity,Tesco,LUID_00003H3N,10.00,10.00,107.00,100.00,-10.00,-7.00,-3.00,[]
1,Cash,GBP,CCY_GBP,-110.00,1.00,-110.00,-110.00,0.00,0.00,0.00,[]


# Examine impact on A2B report

**Note**: The market data in the valuation section must be loaded.

In [38]:
def create_a2b(port, start, end, show_summary):
    if end == "today":
        end = datetime.today().strftime('%Y-%m-%dT23:59:59Z')

    try:
        a2b_report = transaction_portfolios_api.get_a2_b_data(
            scope = module_scope,
            code = f"{module_code}-{port}",
            from_effective_at = start,
            to_effective_at = end,
            recipe_id_scope = module_scope,
            recipe_id_code = f"{module_code}-SimpleStatic",
            property_keys=["Instrument/default/Name"]
        )
    
        a2b_report_df = lusid_response_to_data_frame(a2b_report)
        if show_summary == True:
            a2b_report_df = a2b_report_df.fillna(0)
            a2b_report_df = a2b_report_df.groupby("properties.Instrument/default/Name.value.label_value", as_index=False).sum(numeric_only=True)
            a2b_report_df = a2b_report_df.fillna("")
            # Retain only columns reporting portfolio currency totals
            a2b_report_df = a2b_report_df.filter(items=[
                "properties.Instrument/default/Name.value.label_value",
                "start.portfolio_currency.total",
                "flows.portfolio_currency.total",
                "gains.portfolio_currency.total",
                "carry.portfolio_currency.total",
                "end.portfolio_currency.total"
            ]
            )
        
            # Rename columns
            a2b_report_df.rename(columns = {
                "properties.Instrument/default/Name.value.label_value": "Instrument",
                "start.portfolio_currency.total": "Market Value (Start)",
                "flows.portfolio_currency.total": "Flows",
                "gains.portfolio_currency.total": "CapGains",
                "carry.portfolio_currency.total": "Carry",
                "end.portfolio_currency.total": "Market Value (End)",
            }, inplace = True,
            )
            a2b_report_df.sort_values(by=['Instrument'], inplace=True, ascending = False)
            display(a2b_report_df)
        else:
            display(a2b_report_df.transpose())
    except ApiException as e:
        print(e.body)

In [39]:
for port in ports:
    print(f"\n   {port} portfolio:\n")
    create_a2b(port, "2025-12-31T23:59:59Z", "today", True)


   AllCap portfolio:



,Instrument,Flows,CapGains,Market Value (End)
1,Tesco,110.00,-10.00,100.00
0,GBP,-110.00,0.00,-110.00



   AllNonCap portfolio:



,Instrument,Flows,Carry,Market Value (End)
1,Tesco,110.00,-10.00,100.00
0,GBP,-110.00,0.00,-110.00



   Mixed portfolio:



,Instrument,Flows,CapGains,Carry,Market Value (End)
1,Tesco,110.00,-7.00,-3.00,100.00
0,GBP,-110.00,0.00,0.00,-110.00


# Examine impact on fund accounting

**Note**: The market data in the valuation section must be loaded.

## Set up a simple fund

See https://support.lusid.com/docs/setting-up-a-fund-a-checklist

In [40]:
def create_coa():
    coa_request = lm.ChartOfAccountsRequest(
        code = module_code,
        display_name = f"{module_scope}/{module_code} CoA",
    )
    
    try:
        create_coa_response = chart_accts_api.create_chart_of_accounts(
            scope = module_scope,
            chart_of_accounts_request=coa_request
        )
        print("Success")
    except ApiException as e:
        print(e.body)
        
create_coa()

Success


In [41]:
def add_accounts():
    try:
        add_accounts_response = chart_accts_api.upsert_accounts(
           scope = module_scope,
            code = module_code,
            account = [
                lm.Account(
                    code = "1-Investments",
                    description="Investment account",
                    type = "Asset",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "2-Cash",
                    description="Cash account",
                    type = "Liabilities",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "3-Subscriptions",
                    description="Subscriptions account",
                    type = "Capital",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "4-Redemptions",
                    description="Redemptions account",
                    type = "Capital",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "5-PnL",
                    description="P&L account",
                    type = "Revenue",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "6-YearEnd",
                    description="Cleardown account",
                    type = "Revenue",
                    status = "Active",
                    control = "Manual",
                ),
                lm.Account(
                    code = "7-Error",
                    description="Error account",
                    type = "Revenue",
                    status = "Active",
                    control = "Manual",
                ),
            ]
        )
        print("Success")
    except ApiException as e:
        print(e.body)

add_accounts()

Success


In [42]:
def examine_coa():
    
    coa_response = chart_accts_api.list_accounts(
        scope = module_scope, 
        code = module_code,
        # Retrieve properties to make results more intuitive
        #property_keys = ["Instrument/default/Name"],
    )

    try:
        coa_df = lusid_response_to_data_frame(coa_response)
        # Drop some noisy columns
        coa_df.drop(columns=["control"], inplace=True)
        display(coa_df)
    except ApiException as e:
        print(e.body)
        
examine_coa()

,code,description,type,status,properties
0,1-Investments,Investment account,Asset,Active,{}
1,2-Cash,Cash account,Liabilities,Active,{}
2,3-Subscriptions,Subscriptions account,Capital,Active,{}
3,4-Redemptions,Redemptions account,Capital,Active,{}
4,5-PnL,P&L account,Revenue,Active,{}
5,6-YearEnd,Cleardown account,Revenue,Active,{}
6,7-Error,Error account,Revenue,Active,{}


In [43]:
def add_posting_module():
    pm_request = lm.PostingModuleRequest(
        code = module_code,
        display_name = f"{module_scope}/{module_code} posting module",
        rules = [
            lm.PostingModuleRule(
                rule_id = "rule_1",
                generalLedgerAccountCode = "3-Subscriptions",
                rule_filter = "EconomicBucket startswith 'CA' and Transaction.type eq 'FundsInWithCapitalMovement'"
            ),
            lm.PostingModuleRule(
                rule_id = "rule_2",
                generalLedgerAccountCode = "4-Redemptions",
                rule_filter = "EconomicBucket startswith 'CA' and Transaction.type eq 'FundsOutWithCapitalMovement'"
            ),
            lm.PostingModuleRule(
                rule_id = "rule_3",
                generalLedgerAccountCode = "1-Investments",
                rule_filter = "HoldType eq 'P' and EconomicBucket startswith 'NA'"
            ),
            # Assign tricksy P&L JE Lines to error account first
            lm.PostingModuleRule(
                rule_id = "rule_4",
                generalLedgerAccountCode = "7-Error",
                rule_filter = "EconomicBucket eq 'PL_Other' or EconomicBucket eq 'PL_Rounding'"
            ),
            # Assign legit P&L JE lines
            lm.PostingModuleRule(
                rule_id = "rule_5",
                generalLedgerAccountCode = "5-PnL",
                rule_filter = "EconomicBucket startswith 'PL'"
            ),
            lm.PostingModuleRule(
                rule_id = "rule_6",
                generalLedgerAccountCode = "2-Cash",
                rule_filter = "HoldType neq 'P' and EconomicBucket startswith 'NA'"
            ),
            # Assign unmatched JE lines to error account, if any
            lm.PostingModuleRule(
                rule_id = "rule_7",
                generalLedgerAccountCode = "7-Error",
                rule_filter = "True"
            )
        ]
    )
    
    try:
        pm_response = chart_accts_api.create_posting_module(
            scope = module_scope,
            code = module_code,
            posting_module_request=pm_request
        )
        print("Success")
    except ApiException as e:
        print(e.body)

add_posting_module()

Success


In [44]:
def examine_posting_module(code):
    
    pm_response = chart_accts_api.list_posting_module_rules(
        scope = module_scope, 
        code = module_code,
        posting_module_code = code
    )
    
    pm_response_df = lusid_response_to_data_frame(pm_response)
    display(pm_response_df)
    
examine_posting_module(module_code)

,rule_id,general_ledger_account_code,rule_filter
0,rule_1,3-Subscriptions,EconomicBucket startswith 'CA' and Transaction.type eq 'FundsInWithCapitalMovement'
1,rule_2,4-Redemptions,EconomicBucket startswith 'CA' and Transaction.type eq 'FundsOutWithCapitalMovement'
2,rule_3,1-Investments,HoldType eq 'P' and EconomicBucket startswith 'NA'
3,rule_4,7-Error,EconomicBucket eq 'PL_Other' or EconomicBucket eq 'PL_Rounding'
4,rule_5,5-PnL,EconomicBucket startswith 'PL'
5,rule_6,2-Cash,HoldType neq 'P' and EconomicBucket startswith 'NA'
6,rule_7,7-Error,True


In [45]:
# Create pricing template (has to exist before fund entity)
def create_fund_config():
    create_fundconfig_request = lm.FundConfigurationRequest(
        code = module_code,
        display_name = f"{module_scope}/{module_code} pricing template",
        dealingFilters = [
            lm.ComponentFilter(
                filterId="SUBS",
                filter="generalLedgerAccountCode eq '3-Subscriptions'"
            ),
            lm.ComponentFilter(
                filterId="REDS",
                filter="generalLedgerAccountCode eq '4-Redemptions'"
            ),
        ],
        pnlFilters=[
            lm.ComponentFilter(
                filterId="PNL",
                filter="generalLedgerAccountCode eq '5-PnL'"
            ),
        ],
        backOutFilters=[],
        # externalFeeFilters=[]
    )
    
    try:
        create_fundconfig_response = fundconfig_api.create_fund_configuration(
            scope = module_scope,
            fund_configuration_request=create_fundconfig_request
        )
        print("Success")
    except ApiException as e:
        print(e.body)

create_fund_config()

Success


In [46]:
def create_fund(ibor, ccy):
    
    create_fund_request = lm.FundDefinitionRequest(
        code = f"{module_code}-{ibor}",
        display_name = f"{module_code}-{ibor} fund",
        baseCurrency = ccy,
        portfolio_ids = [
            lm.PortfolioEntityId(
                scope = module_scope,
                code = f"{module_code}-{ibor}"
            )
        ],
        fundConfigurationId=lm.ResourceId(
            scope = module_scope,
            code = module_code
        ),
        investorStructure="NonUnitised",
        type = "Standalone",
        inceptionDate=to_date("2024-01-01"),  # Same as portfolios
        decimalPlaces=5,
        primaryNavType = lm.NavTypeDefinition(
            code = "OFFICIAL",
            displayName="Official NAV",
            description="This is the Official primary NAV type",
            valuationRecipeId=lm.ResourceId(
                scope = module_scope,
                code = f"{module_code}-SimpleStatic"
            ),
            holdingRecipeId=lm.ResourceId(
                scope = module_scope,
                code = f"{module_code}-SimpleStatic"
            ),
            accountingMethod = "AverageCost",
            amortisationMethod = "NoAmortisation",
            chart_of_accounts_id = lm.ResourceId(
                scope = module_scope,
                code = module_code
            ),
            posting_module_codes = [module_code],
            #cleardownModuleCodes=[module_code],
            cashGainLossCalculationDate="SettlementDate",
            # Has to be same as portfolios
            transactionTypeScope=f"{module_scope}{module_code}",
            transactionTemplateScope="default"
        )
    )
    
    try:
        create_fund_response = fund_api.create_fund_v2(
            scope = module_scope,
            fund_definition_request=create_fund_request
        )
        #print(create_fund_response)
        print(f"Fund with display name '{create_fund_response.display_name}' created")
    except ApiException as e:
        print(e.body)

for port in ports:
    create_fund(port, "GBP")

Fund with display name 'TransactionFeeEngine1-AllCap fund' created
Fund with display name 'TransactionFeeEngine1-AllNonCap fund' created
Fund with display name 'TransactionFeeEngine1-Mixed fund' created


## Examine JE Lines

In [47]:
def get_je_lines(ibor, dateordiary, show_summary):
    if dateordiary == "today":
        dateordiary = datetime.today().strftime('%Y-%m-%dT23:59:59Z')
    
    try:
        jelines_response = fund_api.get_valuation_point_journal_entry_lines(
            scope = module_scope,
            code = f"{module_code}-{ibor}",
            valuation_point_data_query_parameters = lm.ValuationPointDataQueryParameters(end=lm.DateOrDiaryEntry(date=dateordiary)),
            nav_type_code="OFFICIAL"
            #filter = "sourceType eq 'LusidTransaction'"
        )
        jelines_response_df = lusid_response_to_data_frame(jelines_response)
        jelines_response_df.sort_values(by=['accounting_date'], inplace=True)
        if show_summary == True:
            jelines_response_df[["accounting_date", "instrument_id", "general_ledger_account_code", 
                                 "base.amount", "source_type", "movement_name", "holding_type", "economic_bucket", "ledger_column"]]
            jelines_response_df = jelines_response_df.filter(items=[
                "accounting_date",
                "instrument_id",
                "general_ledger_account_code",
                "base.amount",
                "source_type",
                "movement_name",
                "holding_type",
                "economic_bucket",
                "ledger_column"
            ])
            display(jelines_response_df)
        else:
            display(jelines_response_df)
    except ApiException as e:
        print(e)

In [48]:
for port in ports:
    print(f"\n{port} fund:\n")
    get_je_lines(port, "today", True)
    # get_je_lines(port, "today", False)


AllCap fund:



,accounting_date,instrument_id,general_ledger_account_code,base.amount,source_type,movement_name,holding_type,economic_bucket,ledger_column
0,2026-01-01 00:00:00+00:00,LUID_00003H3N,1-Investments,110.00,LusidTransaction,Side1,P,NA_Cost,Debit
1,2026-01-01 00:00:00+00:00,CCY_GBP,2-Cash,-110.00,LusidTransaction,Side2,C,NA_Cost,Credit
2,2026-01-05 00:00:00+00:00,CCY_GBP,2-Cash,110.00,LusidTransaction,Side2,C,NA_Cost,Debit
3,2026-01-05 00:00:00+00:00,CCY_GBP,2-Cash,-110.00,LusidTransaction,Side2,B,NA_Cost,Credit
4,2026-07-21 23:59:59+00:00,LUID_00003H3N,1-Investments,-10.00,LusidValuation,MarkToMarket,P,NA_UnrealPriceGL,Credit
5,2026-07-21 23:59:59+00:00,LUID_00003H3N,5-PnL,10.00,LusidValuation,MarkToMarket,P,PL_UnrealPriceGL,Debit



AllNonCap fund:



,accounting_date,instrument_id,general_ledger_account_code,base.amount,source_type,movement_name,holding_type,economic_bucket,ledger_column
0,2026-01-01 00:00:00+00:00,LUID_00003H3N,1-Investments,100.00,LusidTransaction,Side1,P,NA_Cost,Debit
1,2026-01-01 00:00:00+00:00,LUID_00003H3N,5-PnL,10.00,LusidTransaction,CarryNonCapFees,P,PL_Carry,Debit
2,2026-01-01 00:00:00+00:00,CCY_GBP,2-Cash,-110.00,LusidTransaction,Side2,C,NA_Cost,Credit
3,2026-01-05 00:00:00+00:00,CCY_GBP,2-Cash,110.00,LusidTransaction,Side2,C,NA_Cost,Debit
4,2026-01-05 00:00:00+00:00,CCY_GBP,2-Cash,-110.00,LusidTransaction,Side2,B,NA_Cost,Credit



Mixed fund:



,accounting_date,instrument_id,general_ledger_account_code,base.amount,source_type,movement_name,holding_type,economic_bucket,ledger_column
0,2026-01-01 00:00:00+00:00,LUID_00003H3N,1-Investments,107.00,LusidTransaction,Side1,P,NA_Cost,Debit
1,2026-01-01 00:00:00+00:00,LUID_00003H3N,5-PnL,3.00,LusidTransaction,CarryNonCapFees,P,PL_Carry,Debit
2,2026-01-01 00:00:00+00:00,CCY_GBP,2-Cash,-110.00,LusidTransaction,Side2,C,NA_Cost,Credit
3,2026-01-05 00:00:00+00:00,CCY_GBP,2-Cash,110.00,LusidTransaction,Side2,C,NA_Cost,Debit
4,2026-01-05 00:00:00+00:00,CCY_GBP,2-Cash,-110.00,LusidTransaction,Side2,B,NA_Cost,Credit
5,2026-07-21 23:59:59+00:00,LUID_00003H3N,1-Investments,-7.00,LusidValuation,MarkToMarket,P,NA_UnrealPriceGL,Credit
6,2026-07-21 23:59:59+00:00,LUID_00003H3N,5-PnL,7.00,LusidValuation,MarkToMarket,P,PL_UnrealPriceGL,Debit


## Examine trial balance

In [49]:
def get_tb(ibor, dateordiary):
    if dateordiary == "today":
        dateordiary = datetime.today().strftime('%Y-%m-%dT23:59:59Z')
        
    try:
        tb_response = fund_api.get_valuation_point_trial_balance(
            scope = module_scope,
            code = f"{module_code}-{ibor}",
            valuation_point_data_query_parameters = lm.ValuationPointDataQueryParameters(end=lm.DateOrDiaryEntry(date=dateordiary)),
            nav_type_code="OFFICIAL"
        )

        #print(tb_response)
        tb_df = pd.json_normalize(tb_response.to_dict()["values"], sep='.').fillna('')
        tb_df.drop(columns=["levels", "localCurrency", "opening.localAmount", "closing.localAmount", "debit.localAmount", "credit.localAmount"], inplace=True)
        tb_df.rename(columns={'generalLedgerAccountCode': "account",
                              'opening.baseAmount': 'open', 
                              'closing.baseAmount': 'close',
                              'debit.baseAmount': 'DR', 
                              'credit.baseAmount': 'CR'
                             }, inplace=True)

        tb_df = (
                tb_df.assign(
                  order=1,
                  section=np.where(tb_df.accountType.isin({'Asset','Liabilities'}),1,2)
                  )
                  .pipe(lambda tb_df : pd.concat([
                      tb_df,
#                      tb_df.groupby('section',as_index=False)
#                         .sum(numeric_only=True)
#                         .assign(account='Subtotal',
#                                 order=2),
                      tb_df.sum(numeric_only=True)
                        .to_frame()
                        .T
                        .assign(account='Grand Total',
                                order=3)
                      ]))
                  .fillna('')
                  .sort_values(['section','order','account'])
                  .drop(['section','order'],axis=1)
             )

        # # #tb_df = (tb_df.set_index(['general_ledger_account_code','description', 'account_type', 'level 1', 'level 2', 'level 3', 'level 4'])[['opening', 'debit', 'credit', 'closing']])
        tb_df = (tb_df.set_index(['account','description', 'accountType'])[['open','DR','CR', 'close']])
        display(tb_df)
    except ApiException as e:
        print(e.body)

In [50]:
for port in ports:
    print(f"\n{port} fund:")
    get_tb(port, "today")


AllCap fund:


,,,open,DR,CR,close
account,description,accountType,,,,
1-Investments,Investment account,Asset,0.00,110.00,-10.00,100.00
2-Cash,Cash account,Liabilities,0.00,110.00,-220.00,-110.00
5-PnL,P&L account,Revenue,0.00,10.00,0.00,10.00
Grand Total,,,0.00,230.00,-230.00,0.00



AllNonCap fund:


,,,open,DR,CR,close
account,description,accountType,,,,
1-Investments,Investment account,Asset,0.00,100.00,0.00,100.00
2-Cash,Cash account,Liabilities,0.00,110.00,-220.00,-110.00
5-PnL,P&L account,Revenue,0.00,10.00,0.00,10.00
Grand Total,,,0.00,220.00,-220.00,0.00



Mixed fund:


,,,open,DR,CR,close
account,description,accountType,,,,
1-Investments,Investment account,Asset,0.00,107.00,-7.00,100.00
2-Cash,Cash account,Liabilities,0.00,110.00,-220.00,-110.00
5-PnL,P&L account,Revenue,0.00,10.00,0.00,10.00
Grand Total,,,0.00,227.00,-227.00,0.00
